# Task 1 - Full Feature Preparation Pipeline

This notebook uses the messy student dataset and applies the required preprocessing steps.

In [48]:
!pip install scikit-learn


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 1: Import Required Libraries

In [64]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

### Step 2: Set Up File Paths

Define where to read the messy data and where to save cleaned outputs.

In [50]:
BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / 'messy_students_data.csv'
OUTPUT_NOTES_PATH = BASE_DIR / 'task1_prep_notes.txt'

print(f"Data will be loaded from: {DATA_PATH}")
print(f"Outputs will be saved to: {BASE_DIR}")

Data will be loaded from: c:\Users\PM\Desktop\Internship\AI-Data-Engineering-Internship\Week-7 Jun_2\messy_students_data.csv
Outputs will be saved to: c:\Users\PM\Desktop\Internship\AI-Data-Engineering-Internship\Week-7 Jun_2


### Step 3: Define Helper Functions

These functions clean up the messy data. We'll define one function for each cleaning task.

In [51]:
def normalize_text(value):
    """
    Remove extra quotes and whitespace from a value.
    Example: \"'  hello  '\" becomes "hello"
    """
    if pd.isna(value):
        return np.nan
    cleaned_value = str(value).strip()
    cleaned_value = cleaned_value.strip('"')
    cleaned_value = cleaned_value.strip("'")
    cleaned_value = re.sub(r'\s+', ' ', cleaned_value)
    return cleaned_value

# Test the function
test_messy = '  \"hello\"  '
print(f"Original: {test_messy}")
print(f"Cleaned: {normalize_text(test_messy)}")

Original:   "hello"  
Cleaned: hello


In [52]:
def extract_numeric_value(value):
    """
    Extract the first number from a messy cell like "28 marks" → 28.0
    """
    if pd.isna(value):
        return np.nan
    cleaned_value = normalize_text(value)
    if pd.isna(cleaned_value):
        return np.nan
    match = re.search(r'-?\d+(?:\.\d+)?', str(cleaned_value))
    if match is None:
        return np.nan
    return float(match.group())

# Test the function
test_score = "28 marks"
print(f"Original: {test_score}")
print(f"Extracted number: {extract_numeric_value(test_score)}")

Original: 28 marks
Extracted number: 28.0


In [53]:
def clean_grade_level(value):
    """
    Convert grade strings to numeric level.
    Example: "Grade 5" → 5.0, "03" → 3.0
    """
    if pd.isna(value):
        return np.nan
    normalized_value = normalize_text(value)
    if pd.isna(normalized_value):
        return np.nan
    match = re.search(r'\d+', str(normalized_value))
    if match is None:
        return np.nan
    return float(match.group())

# Test the function
test_grades = ['Grade 5', '03', '11', 'Grade 12']
for g in test_grades:
    print(f"{g:15} → {clean_grade_level(g)}")

Grade 5         → 5.0
03              → 3.0
11              → 11.0
Grade 12        → 12.0


In [54]:
def clean_gender(value):
    """
    Standardize gender to 'male' or 'female'.
    Treat numeric codes (0, 1) as missing values (NaN).
    """
    if pd.isna(value):
        return np.nan
    normalized_value = normalize_text(value)
    if pd.isna(normalized_value):
        return np.nan
    lower_value = str(normalized_value).lower()
    mapping = {'m': 'male', 'male': 'male', 'f': 'female', 'female': 'female'}
    return mapping.get(lower_value, np.nan)

# Test the function
test_genders = ['M', 'Female', 'f', 'm', '0', '1']
for g in test_genders:
    print(f"{g:10} → {clean_gender(g)}")

M          → male
Female     → female
f          → female
m          → male
0          → nan
1          → nan


In [55]:
raw_data = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {raw_data.shape}")
print(f"\nColumn names: {list(raw_data.columns)}")
print(f"\nFirst few rows:")
print(raw_data.head(3))

Dataset shape: (1000, 7)

Column names: ['Name', 'Gender', 'Grade', 'Math', 'Science', 'English', 'Total']

First few rows:
      Name Gender     Grade      Math   Science English  Total
0  'Navya'   male        11      47.0      63.0    74.0    184
1    ROHAN      F   Grade 3      16.0        77       8    101
2  'Aditi'      0  Grade 11  28 marks  43 marks    46.0    117


### Step 4: Load the Raw Data

Load the messy CSV file into a Pandas DataFrame and inspect it.

In [56]:
cleaned_data = pd.DataFrame(
    {
        'Gender': raw_data['Gender'].map(clean_gender),
        'grade_level': raw_data['Grade'].map(clean_grade_level),
        'Math': raw_data['Math'].map(extract_numeric_value),
        'Science': raw_data['Science'].map(extract_numeric_value),
        'English': raw_data['English'].map(extract_numeric_value),
        'Total': raw_data['Total'].map(extract_numeric_value),
    }
)

print("Data cleaned!")
print(f"Shape: {cleaned_data.shape}")
print(f"\nFirst few rows (now clean):")
print(cleaned_data.head(3))
print(f"\nData types:")
print(cleaned_data.dtypes)
print(f"\nMissing values per column:")
print(cleaned_data.isnull().sum())

Data cleaned!
Shape: (1000, 6)

First few rows (now clean):
   Gender  grade_level  Math  Science  English  Total
0    male         11.0  47.0     63.0     74.0  184.0
1  female          3.0  16.0     77.0      8.0  101.0
2     NaN         11.0  28.0     43.0     46.0  117.0

Data types:
Gender             str
grade_level    float64
Math           float64
Science        float64
English        float64
Total          float64
dtype: object

Missing values per column:
Gender         155
grade_level      0
Math             0
Science          0
English          0
Total            0
dtype: int64


### Step 5: Clean and Standardize the Data

Apply our helper functions to clean each column. We drop the 'Name' column since it's just an identifier with high cardinality.

In [57]:
# Drop rows where the target (Total) is missing
cleaned_data = cleaned_data.dropna(subset=['Total'])

# Separate features (X) and target (y)
feature_frame = cleaned_data.drop(columns=['Total'])
target_series = cleaned_data['Total']

# Ensure correct column order
feature_frame = feature_frame[['Gender', 'grade_level', 'Math', 'Science', 'English']]

print(f"Features shape: {feature_frame.shape}")
print(f"Target shape: {target_series.shape}")
print(f"\nFeature columns: {list(feature_frame.columns)}")
print(f"Target range: {target_series.min():.1f} to {target_series.max():.1f}")

Features shape: (1000, 5)
Target shape: (1000,)

Feature columns: ['Gender', 'grade_level', 'Math', 'Science', 'English']
Target range: 13.0 to 284.0


### Step 6: Prepare Features and Target

Remove rows with missing targets, separate features (X) from target (y).

In [58]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    feature_frame,
    target_series,
    test_size=0.2,
    random_state=42
)

print(f"✓ Split complete!")
print(f"Training set: {X_train_raw.shape[0]} samples (80%)")
print(f"Test set: {X_test_raw.shape[0]} samples (20%)")
print(f"Total: {X_train_raw.shape[0] + X_test_raw.shape[0]} samples")

✓ Split complete!
Training set: 800 samples (80%)
Test set: 200 samples (20%)
Total: 1000 samples


### Step 7: Split Data into Train and Test Sets

Use 80% for training and 20% for testing. This ensures the model sees new data during evaluation.

In [59]:
def build_preparation_pipeline():
    """
    Create the full preprocessing pipeline.
    Returns a ColumnTransformer that applies different transformations to different columns.
    """
    numeric_features = ['grade_level', 'Math', 'Science', 'English']
    categorical_features = ['Gender']

    # Pipeline for numeric columns
    numeric_pipeline = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]
    )

    # Pipeline for categorical columns
    categorical_pipeline = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]
    )

    # Combine both pipelines
    preprocessor = ColumnTransformer(
        transformers=[
            ('numeric', numeric_pipeline, numeric_features),
            ('categorical', categorical_pipeline, categorical_features),
        ],
        remainder='drop',
        verbose_feature_names_out=False,
    )
    
    return preprocessor

preprocessor = build_preparation_pipeline()
print("✓ Preprocessing pipeline created successfully!")

✓ Preprocessing pipeline created successfully!


### Step 8: Build the Preprocessing Pipeline

Create a pipeline that:
- Imputes numeric columns with median values
- Imputes categorical columns with the most frequent value  
- One-hot encodes categorical variables
- Scales numeric variables with StandardScaler

In [60]:
# Fit on training data, then transform
X_train_processed = preprocessor.fit_transform(X_train_raw)
X_test_processed = preprocessor.transform(X_test_raw)

# Get feature names from the pipeline
processed_feature_names = preprocessor.get_feature_names_out()

# Convert to DataFrames for easier inspection
X_train = pd.DataFrame(X_train_processed, columns=processed_feature_names, index=X_train_raw.index)
X_test = pd.DataFrame(X_test_processed, columns=processed_feature_names, index=X_test_raw.index)

print("✓ Preprocessing applied!")
print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"\nFeature names (after encoding & scaling):")
for i, name in enumerate(processed_feature_names, 1):
    print(f"  {i}. {name}")
print(f"\nFirst row of X_train (scaled values):")
print(X_train.iloc[0])

✓ Preprocessing applied!

X_train shape: (800, 6)
X_test shape: (200, 6)

Feature names (after encoding & scaling):
  1. grade_level
  2. Math
  3. Science
  4. English
  5. Gender_female
  6. Gender_male

First row of X_train (scaled values):
grade_level      1.058201
Math             0.716636
Science         -0.557222
English          0.367726
Gender_female    1.000000
Gender_male      0.000000
Name: 29, dtype: float64


### Step 9: Apply the Preprocessing Pipeline

Fit the pipeline on the training data, then transform both training and test data.

In [61]:
notes = [
    '1. Dropped Name because it is an identifier-style column with very high cardinality and no stable predictive value.',
    '2. Kept Total as the target so the feature pipeline can clean and prepare the remaining student attributes.',
    '3. Standardized Gender into male/female and treated numeric codes such as 0 and 1 as missing because they are not meaningful labels.',
    '4. Converted Grade into an ordinal grade_level feature so the natural order of grades is preserved instead of being flattened into arbitrary categories.',
    "5. Parsed Math, Science, English, and Total from dirty text values like '28 marks' into numeric columns before preprocessing.",
    '6. Used median imputation for numeric columns because it is robust to outliers and works well on the noisy score values.',
    '7. Used most_frequent imputation for Gender because the column is categorical and the mode is the safest default fill.',
    '8. One-hot encoded Gender because it is nominal and should not be forced into an artificial order.',
    '9. Applied StandardScaler to the numeric feature block so grade_level and the subject scores are on comparable scales.',
    '10. Performed an 80/20 train-test split to keep a clean holdout set for later modeling.',
]

print("PREPROCESSING DECISIONS:\n")
for note in notes:
    print(note)
    print()

PREPROCESSING DECISIONS:

1. Dropped Name because it is an identifier-style column with very high cardinality and no stable predictive value.

2. Kept Total as the target so the feature pipeline can clean and prepare the remaining student attributes.

3. Standardized Gender into male/female and treated numeric codes such as 0 and 1 as missing because they are not meaningful labels.

4. Converted Grade into an ordinal grade_level feature so the natural order of grades is preserved instead of being flattened into arbitrary categories.

5. Parsed Math, Science, English, and Total from dirty text values like '28 marks' into numeric columns before preprocessing.

6. Used median imputation for numeric columns because it is robust to outliers and works well on the noisy score values.

7. Used most_frequent imputation for Gender because the column is categorical and the mode is the safest default fill.

8. One-hot encoded Gender because it is nominal and should not be forced into an artificial

### Step 10: Document Preprocessing Decisions

Write down the reasoning behind each choice we made.

### Step 11: Save the Outputs

Save X_train, X_test, y_train, y_test as CSV files, and the decision notes as a text file.

### Summary

You now have 4 clean datasets ready for machine learning:
- **X_train.csv**: Features for training (80% of data)
- **X_test.csv**: Features for testing (20% of data)
- **y_train.csv**: Target values for training
- **y_test.csv**: Target values for testing

All missing values have been imputed, categoricals have been encoded, and numerics have been scaled to be on the same scale!

In [62]:
# Save the processed data as CSV files
X_train.to_csv(BASE_DIR / 'X_train.csv', index=False)
X_test.to_csv(BASE_DIR / 'X_test.csv', index=False)
y_train.to_csv(BASE_DIR / 'y_train.csv', index=False)
y_test.to_csv(BASE_DIR / 'y_test.csv', index=False)

# Save the decision notes
OUTPUT_NOTES_PATH.write_text('\n'.join(notes) + '\n', encoding='utf-8')

print("✓ All files saved successfully!\n")
print(f"Saved files:")
print(f"  • {BASE_DIR / 'X_train.csv'}")
print(f"  • {BASE_DIR / 'X_test.csv'}")
print(f"  • {BASE_DIR / 'y_train.csv'}")
print(f"  • {BASE_DIR / 'y_test.csv'}")
print(f"  • {OUTPUT_NOTES_PATH}")

✓ All files saved successfully!

Saved files:
  • c:\Users\PM\Desktop\Internship\AI-Data-Engineering-Internship\Week-7 Jun_2\X_train.csv
  • c:\Users\PM\Desktop\Internship\AI-Data-Engineering-Internship\Week-7 Jun_2\X_test.csv
  • c:\Users\PM\Desktop\Internship\AI-Data-Engineering-Internship\Week-7 Jun_2\y_train.csv
  • c:\Users\PM\Desktop\Internship\AI-Data-Engineering-Internship\Week-7 Jun_2\y_test.csv
  • c:\Users\PM\Desktop\Internship\AI-Data-Engineering-Internship\Week-7 Jun_2\task1_prep_notes.txt
